# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [26]:
!pip install openai

In [27]:
import logging
import os
import re
import json
import logging
import time
from datetime import datetime
from openai import OpenAI
from IPython.display import display, Markdown, clear_output

# Configure basic logging for monitoring the pipeline. Every major action performed by the agent is recorded. (Examples: Query received, Tool selected, Errors, Routing decisions)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

- `re` → Pattern matching
- `json` → Structured outputs
- `logging` → Execution tracking
- `OpenAI` → LLM access
- `IPython.display` → Better UI rendering

# Retrieve OpenRouter API securely and Create OpenAI client using OpenRouter.
- The client becomes the communication interface between our application and the LLM.
- Note: Never expose API keys directly inside source code. Use Google Colab Secrets.

In [16]:
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("Free-Models-Router")
    logging.info("✅ API key retrieved securely from Google Colab Secrets.")
except Exception as e:
    # Fallback to standard environment variable if not running in Colab
    OPENROUTER_API_KEY = os.environ.get("Free Models Router")
    logging.warning("⚠️ Could not load Google Colab userdata. Falling back to environment variables.")

# Initialize the OpenRouter OpenAI client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-OpenRouter-Title": "Colab Agentic Pipeline Project",
    }
)

# Use a specific, fast, free model via OpenRouter's free tier routing
LLM_MODEL = "openrouter/free"

- The client abstracts HTTP requests and simplifies interaction with external AI models.

# Tool Development

### TOOL 1: Calculator

In [28]:
# 🛠️ TOOL 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # Strip out non-numeric/operator words if they passed through routing
        clean_expr = expression.replace("calculate", "").strip()
        return str(eval(clean_expr))
    except Exception as e:
        logging.error(f"Calculator failed for expression '{expression}': {e}")
        return "Error in calculation"

### TOOL 2: Keyword Extractor

In [29]:
# 🛠️ TOOL 2: Keyword Extractor
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        # Strip out the trigger word to avoid extracting it as a keyword
        clean_text = text.replace("extract keywords from", "").replace("keywords", "").strip()
        words = clean_text.split()
        # Clean punctuation and filter words longer than 4 characters
        cleaned_words = [w.strip(".,!?;:()").lower() for w in words]
        keywords = list(set([w for w in cleaned_words if len(w) > 4]))
        return keywords[:5]
    except Exception as e:
        logging.error(f"Keyword extraction failed for text '{text}': {e}")
        return []

- Remove trigger words
- Split sentence
- Clean punctuation
- Filter words
- Remove duplicates

### Tool 3 — Currency Converter

In [30]:
# Tool 3 (NEW): Currency Converter (Mock Live Data): Convert currencies using predefined exchange rates.
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert currencies using standard fixed benchmark mock rates."""
    # Mock conversion rates against USD base
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "INR": 83.5}
    try:
        from_rate = rates.get(from_currency.upper())
        to_rate = rates.get(to_currency.upper())
        if not from_rate or not to_rate:
            return f"Unsupported currency code. Supported: {list(rates.keys())}"

        usd_amount = amount / from_rate
        converted = usd_amount * to_rate
        return f"{amount} {from_currency.upper()} = {round(converted, 2)} {to_currency.upper()}"
    except Exception as e:
        return f"Currency conversion error: {str(e)}"

### Logic

```
Amount
↓
Base Currency
↓
Target Currency
↓
Conversion Formula
↓
Result
```

### Critical Learning

Shows how external APIs could later replace mock data without changing the agent logic.


### Tool 4 (NEW): Password Generator

In [31]:
# Tool 4 (NEW): Password Generator
def generate_password(length: int = 12) -> str:
    """Generate a secure randomly organized mockup password string."""
    import random
    import string
    try:
        chars = string.ascii_letters + string.digits + "!@#$%^&*"
        length = max(6, min(length, 64))  # Bounds checking between 6 and 64
        return "".join(random.choice(chars) for _ in range(length))
    except Exception:
        return "Failed to generate password."

### Logic

Uses:

- letters
- numbers
- symbols

Randomly selects characters.

### Critical Learning

Utility tools extend agent capability beyond question answering.

### Tool 5 — Text Counter

In [32]:
# Tool 5 (NEW): Word & Character Counter
def string_counter(text: str) -> dict:
    """Count words and characters in a piece of text."""
    try:
        char_count = len(text)
        word_count = len(text.split())
        return {"words": word_count, "characters": char_count}
    except Exception:
        return {"words": 0, "characters": 0}

### Logic

```
Input Text -> Split() -> Count Words -> Count Characters
```

### Critical Learning

Simple utility tools can be integrated into the same agent architecture.

### Tool 6 (NEW): Temperature Converter

In [33]:
# Tool 6 (NEW): Temperature Converter
def convert_temperature(value: float, unit: str) -> str:
    """Convert Celsius to Fahrenheit or vice versa."""
    try:
        unit = unit.upper()
        if 'C' in unit:
            fahrenheit = (value * 9/5) + 32
            return f"{value}°C is {round(fahrenheit, 2)}°F"
        elif 'F' in unit:
            celsius = (value - 32) * 5/9
            return f"{value}°F is {round(celsius, 2)}°C"
        else:
            return "Unknown unit. Use 'C' for Celsius or 'F' for Fahrenheit."
    except Exception:
        return "Temperature parsing failure."

### Logic

Uses mathematical conversion formulas.

### Critical Learning

Rule-based deterministic tools are ideal for fixed calculations.

### Tool 7 (NEW): Palindrome Checker

In [34]:
# Tool 7 (NEW): Palindrome Checker
def check_palindrome(text: str) -> str:
    """Check if a string (or isolated core word) is a valid palindrome."""
    try:
        # Lowercase and strip basic punctuation/trailing spaces
        clean_text = text.lower().strip("?.! ")

        # Smart cleanup: If the sentence still contains filler words, isolate the last or longest word
        # This acts as a guardrail if regex leaves "the word racecar"
        filler_words = {"the", "word", "is", "a", "check", "if", "palindrome"}
        words = [w.strip(".,!?;:") for w in clean_text.split()]
        filtered_words = [w for w in words if w not in filler_words and len(w) > 0]

        # Target the core word to check
        target_word = filtered_words[0] if filtered_words else clean_text

        # Remove any non-alphanumeric characters from the target word for the final mirror test
        final_check_string = "".join(char for char in target_word if char.isalnum())

        if not final_check_string:
            return "Could not extract a valid word to check."

        is_palindrome = final_check_string == final_check_string[::-1]

        return f"'{target_word}' is {'a palindrome' if is_palindrome else 'not a palindrome'}."
    except Exception as e:
        return f"Could not compute palindrome evaluation: {str(e)}"

### Logic

- Clean input
- Remove punctuation
- Reverse string
- Compare

### Critical Learning

Input cleaning greatly improves tool robustness.

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [35]:
# 🤖 AGENT FUNCTION
def agent(query: str) -> dict:
    """
    Single-Agent Router processing queries, activating matching utility tools,
    or falling back seamlessly onto OpenRouter LLM infrastructure.
    """
    query_lower = query.lower()
    logging.info(f"Incoming user query routing: '{query}'")

    try:
        # Route 1: Calculator
        if "calculate" in query_lower or any(op in query for op in ["+", "-", "*", "/"]):
            logging.info("🔀 Routed to: Calculator Tool")
            expr = query_lower.replace("calculate", "").strip()
            return {"type": "calculation", "result": calculator(expr)}

        # Route 2: Keywords Extractor
        elif "keywords" in query_lower or "extract" in query_lower:
            logging.info("🔀 Routed to: Keyword Extractor")
            return {"type": "keywords", "result": extract_keywords(query)}

        # Route 3: Currency Converter
        elif "convert currency" in query_lower or "exchange" in query_lower or " to " in query_lower and any(curr in query_lower for curr in ["usd", "eur", "gbp", "inr"]):
            logging.info("🔀 Routed to: Currency Converter")
            # Primitive semantic extractor for numbers
            numbers = re.findall(r"\d+\.?\d*", query)
            amount = float(numbers[0]) if numbers else 1.0

            # Simple currency code identification
            found_currencies = [c for c in ["USD", "EUR", "GBP", "INR"] if c in query.upper()]
            from_c = found_currencies[0] if len(found_currencies) > 0 else "USD"
            to_c = found_currencies[1] if len(found_currencies) > 1 else "INR"

            return {"type": "currency_conversion", "result": convert_currency(amount, from_c, to_c)}

        # Route 4: Password Generator
        elif "password" in query_lower or "generate clear code" in query_lower:
            logging.info("🔀 Routed to: Password Generator")
            numbers = re.findall(r"\d+", query)
            length = int(numbers[0]) if numbers else 12
            return {"type": "password_generation", "result": generate_password(length)}

        # Route 5: Counter Tool
        elif "count" in query_lower and any(x in query_lower for x in ["word", "character", "length"]):
            logging.info("🔀 Routed to: Text Counter")
            clean_payload = query_lower.replace("count words and characters in", "").replace("count", "").strip()
            return {"type": "text_analysis", "result": string_counter(clean_payload)}

        # Route 6: Temperature Converter
        elif "temperature" in query_lower or "degree" in query_lower:
            logging.info("🔀 Routed to: Temperature Converter")
            numbers = re.findall(r"[-+]?\d*\.?\d+", query)
            value = float(numbers[0]) if numbers else 0.0
            unit = "F" if "fahrenheit" in query_lower or " f " in query_lower or "f " in query_lower else "C"
            return {"type": "temperature_conversion", "result": convert_temperature(value, unit)}

        # Route 7: Palindrome Checker
        elif "palindrome" in query_lower:
            logging.info("🔀 Routed to: Palindrome Checker")

            # Extract anything inside single or double quotes if present
            quoted_match = re.search(r'["\']([^"\']+)["\']', query)
            if quoted_match:
                payload = quoted_match.group(1)
            else:
                # Otherwise, pass the raw string and let the upgraded tool isolate the word
                payload = query

            return {"type": "palindrome_check", "result": check_palindrome(payload)}

        # Route 8: General Knowledge Fallback using OpenRouter API
        else:
            logging.info("🔀 No static route matches. Routing query to OpenRouter LLM...")

            if not OPENROUTER_API_KEY:
                return {
                    "type": "error",
                    "result": "Missing OpenRouter API Key. Configure the Colab secret key 'Free Models Router'."
                }

            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": "You are a helpful, precise AI agent assistant."},
                    {"role": "user", "content": query}
                ],
                max_tokens=250
            )

            llm_output = response.choices[0].message.content.strip()
            return {"type": "general", "result": llm_output}

    except Exception as general_err:
        logging.error(f"Pipeline processing failure: {general_err}")
        return {"type": "error", "result": f"Execution exception context: {str(general_err)}"}

The agent decides:

- Which tool to invoke
- When to call the LLM
- How to structure responses

### Agent Flow

```
Receive Query -> Convert to Lowercase -> Intent Detection -> Conditional Routing -> Execute Tool -> Return JSON
```

---

### Routing Logic

```
Math -> Calculator
----------------
Keywords -> Keyword Tool
----------------
Currency -> Currency Tool
----------------
Password -> Password Tool
----------------
Counter -> Counter Tool
----------------
Temperature -> Temperature Tool
----------------
Palindrome -> Palindrome Tool
----------------
Else -> OpenRouter LLM
```

### Critical Learning

The agent itself performs **reasoning**, while tools perform **execution**.


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [39]:
# 🧪 Test Cases

queries = [
    "Calculate 145 * 6",                                                        # Tool 1
    "Extract keywords from Artificial Intelligence is transforming industries", # Tool 2
    "Convert currency 50 USD to EUR",                                           # Tool 3
    "Generate a secure password with length 16",                               # Tool 4
    "Count the words and characters inside supercalifragilisticexpialidocious", # Tool 5
    "What is the temperature conversion for 38 C?",                            # Tool 6
    "Check if the word racecar is a palindrome",                                # Tool 7
    "Explain quantum computing."                                # OpenRouter LLM
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 145 * 6
Response: {'type': 'calculation', 'result': '870'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'transforming', 'industries', 'artificial', 'extract']}
--------------------------------------------------
Query: Convert currency 50 USD to EUR
Response: {'type': 'currency_conversion', 'result': '50.0 USD = 46.0 EUR'}
--------------------------------------------------
Query: Generate a secure password with length 16
Response: {'type': 'password_generation', 'result': '@Y6%crvIb98C0x79'}
--------------------------------------------------
Query: Count the words and characters inside supercalifragilisticexpialidocious
Response: {'type': 'text_analysis', 'result': {'words': 6, 'characters': 66}}
--------------------------------------------------
Query: What is the temperature conversion for 38 C?
Response: {'type': 'temperature

Note:
- Your last query's output contains the model's reasoning/thinking process as the OpenRouter free router may route your request to different free models. Some reasoning models expose their internal reasoning or chain-of-thought in the output instead of only the final answer.
- Notice that the markdown is still just a string stored in the JSON. The notebook can later render that string as Markdown, as shown in next cell.

In [38]:
print("\n=== STARTING INTERACTIVE EXECUTION MODE ===")
while True:
    try:
        user_input = input("Enter query (type 'exit' to stop): ")

        if user_input.lower() == "exit":
            print("Stopping AI Agent Pipeline. Goodbye!")
            break

        if not user_input.strip():
            continue

        # 1. Get the structured dict response from your agent
        response = agent(user_input)

        print("\n" + "="*50)
        print(f"📊 Response Type: {response['type'].upper()}")
        print("="*50)

        # 2. Render markdown and LaTeX elegantly or print raw tool outputs
        if response['type'] == 'general' and isinstance(response['result'], str):
            display(Markdown(response['result']))
        else:
            # If it's a list or dictionary from a tool, convert it to a string cleanly
            print(json.dumps(response['result'], indent=2))

        print("="*50 + "\n")

        # 3. Small UI flush pause so Colab finishes rendering before showing the next prompt
        time.sleep(0.2)

    except KeyboardInterrupt:
        print("\nSession interrupted manually. Goodbye!")
        break
    except Exception as e:
        print(f"⚠️ Interactive Loop Error: {e}\n")
        time.sleep(0.2)


=== STARTING INTERACTIVE EXECUTION MODE ===
Enter query (type 'exit' to stop): What is third law of thermodynamics

📊 Response Type: GENERAL


**Third Law of Thermodynamics (Law of Absolute Zero)**  

*Statement*  
> As the temperature of a system approaches absolute zero (0 K, –273.15 °C), the entropy of a perfect crystalline substance approaches a constant minimum value, which is conventionally taken to be zero.

*Key points*

| Aspect | Explanation |
|--------|--------------|
| **Absolute zero** | The lowest possible temperature; at 0 K the thermal motion of particles ceases (classically) and quantum mechanically the system is in its ground state. |
| **Entropy (S)** | A measure of the number of microscopic configurations compatible with the macroscopic state. At absolute zero a perfect crystal has only one such configuration, so its entropy is zero. |
| **Third‑law formulation (Nernst‑Planck statement)** | For any process, the change in entropy ΔS → 0 as T → 0 K. In


Enter query (type 'exit' to stop): exit
Stopping AI Agent Pipeline. Goodbye!


# 🏗 Overall Pipeline

```
                 User Query
                      │
                      ▼
             Agent (Router)
                      │
       Intent Detection / Routing
                      │
      ┌───────────────┼────────────────┐
      ▼               ▼                ▼
   Tool Call      Tool Call      OpenRouter LLM
      │               │                │
      └───────────────┴────────────────┘
                      │
                      ▼
           Structured JSON Response
```


# ✅ Conclusion

This project demonstrates the complete workflow of a **Single-Agent Agentic AI Pipeline**. The agent acts as the central decision-maker by analyzing user intent, routing requests to specialized tools when deterministic execution is required, and leveraging an LLM for open-ended reasoning tasks. The modular design, structured JSON outputs, logging, and error handling make the solution organized, extensible, and closer to a production-ready architecture. By extending the assignment with additional tools and an interactive interface, the project showcases key Agentic AI concepts such as intent-based routing, tool integration, modular development, and hybrid rule-based plus LLM-driven decision making.
One suggestion